## Step 0B — NOAA / MarineCadastre AIS Source Investigation

### Source Overview
NOAA's Automatic Identification System (AIS) dataset contains historical vessel-position observations collected from AIS transmissions within the dataset's geographic coverage.

Unlike the USCG Merchant Vessels and PSIX registry sources, AIS is primarily **movement/event data**, with potentially many records per vessel.

- **Source:** NOAA / MarineCadastre
- **Format:** Daily compressed CSV files
- **Expected grain:** One vessel-position observation at a timestamp
- **Key fields:** MMSI, BaseDateTime, LAT, LON, SOG, COG, Heading, IMO, VesselName
- **Population:** U.S. and foreign vessels observed within the dataset coverage

### Investigation Objective
Examine the source's file organization, original schema, data types, identifiers, nulls, and unusual values before designing transformations.

Determine how AIS observations and vessel attributes might complement the USCG registry sources in a future **canonical Gold vessel table**.

**Current scope:** Source discovery and profiling only. No canonical mappings or ingestion pipeline implementation yet.

### 1. Inspect Source File Index
Retrieve the 2026 AIS file listing to examine naming conventions, file organization, and available download formats before selecting a representative sample.

**Source:** https://noaaocm.blob.core.windows.net/ais/csv2/csv2026/index.html

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

index_url = (
    "https://noaaocm.blob.core.windows.net/"
    "ais/csv2/csv2026/index.html"
)

response = requests.get(index_url, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

files = [
    {
        "filename": link.get_text(strip=True),
        "url": urljoin(index_url, link["href"])
    }
    for link in soup.find_all("a", href=True)
    if link["href"].lower().endswith(
        (".csv", ".csv.zst", ".zip", ".zst")
    )
]

print(f"Files found: {len(files)}")

for file in files[:10]:
    print(file["filename"])

Files found: 181
ais-2026-01-01.csv.zst
ais-2026-01-02.csv.zst
ais-2026-01-03.csv.zst
ais-2026-01-04.csv.zst
ais-2026-01-05.csv.zst
ais-2026-01-06.csv.zst
ais-2026-01-07.csv.zst
ais-2026-01-08.csv.zst
ais-2026-01-09.csv.zst
ais-2026-01-10.csv.zst


In [2]:
sample = files[0]

response = requests.head(
    sample["url"],
    allow_redirects=True,
    timeout=30
)
response.raise_for_status()

size_bytes = int(response.headers.get("Content-Length", 0))

print("Filename:", sample["filename"])
print("Size (MB):", round(size_bytes / 1024**2, 2))
print("Content type:", response.headers.get("Content-Type"))

Filename: ais-2026-01-01.csv.zst
Size (MB): 228.16
Content type: application/octet-stream


In [3]:
from pathlib import Path
import requests

raw_dir = Path("../data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

sample_path = raw_dir / sample["filename"]

if sample_path.exists():
    print(f"Already downloaded: {sample_path}")
else:
    with requests.get(sample["url"], stream=True, timeout=60) as response:
        response.raise_for_status()

        with open(sample_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

    print(f"Downloaded: {sample_path}")

print(f"File size: {sample_path.stat().st_size / 1024**2:.2f} MB")

Downloaded: ../data/raw/ais-2026-01-01.csv.zst
File size: 228.16 MB


In [4]:
import zstandard as zstd
import pandas as pd

with open(sample_path, "rb") as f:
    dctx = zstd.ZstdDecompressor()

    with dctx.stream_reader(f) as reader:
        df_sample = pd.read_csv(reader, nrows=5)

display(df_sample)
print("\nColumns:", df_sample.columns.tolist())
print("\nShape:", df_sample.shape)

,mmsi,base_date_time,longitude,latitude,sog,cog,heading,vessel_name,imo,call_sign,vessel_type,status,length,width,draft,cargo,transceiver
0,368164470,2026-01-01 00:00:03,-76.31518,36.84956,0.0,149.6,19.0,EASTERN BRANCH,NaN,WDL7894,52,0.0,20,8,2.3,52.0,A
1,366953930,2026-01-01 00:00:02,-74.20510,40.61131,0.0,250.4,191.0,MATTHEW TIBBETTS,IMO8986949,WDB8156,52,0.0,27,16,4.0,52.0,A
2,367328790,2026-01-01 00:00:01,-97.16752,27.85969,0.0,11.2,34.0,LADY OF PERPETUAL HE,NaN,WDE2720,31,12.0,207,22,3.1,57.0,A
3,368025040,2026-01-01 00:00:04,-122.50573,48.75658,0.0,NaN,NaN,TRUTINA,NaN,WDJ9334,37,NaN,13,4,NaN,NaN,B
4,368284370,2026-01-01 00:00:01,-90.46691,40.00764,4.7,79.2,NaN,DIANE DENISE,NaN,WDN4808,31,12.0,347,33,3.0,57.0,A



Columns: ['mmsi', 'base_date_time', 'longitude', 'latitude', 'sog', 'cog', 'heading', 'vessel_name', 'imo', 'call_sign', 'vessel_type', 'status', 'length', 'width', 'draft', 'cargo', 'transceiver']

Shape: (5, 17)


In [5]:
schema_profile = pd.DataFrame({
    "column": df_sample.columns,
    "pandas_dtype": df_sample.dtypes.astype(str).values,
    "non_null": df_sample.notna().sum().values,
    "null_count": df_sample.isna().sum().values,
    "sample_value": [
        df_sample[col].dropna().iloc[0]
        if df_sample[col].notna().any() else None
        for col in df_sample.columns
    ]
})

display(schema_profile)

,column,pandas_dtype,non_null,null_count,sample_value
0,mmsi,int64,5,0,368164470
1,base_date_time,str,5,0,2026-01-01 00:00:03
2,longitude,float64,5,0,-76.31518
3,latitude,float64,5,0,36.84956
4,sog,float64,5,0,0.0
5,cog,float64,4,1,149.6
6,heading,float64,3,2,19.0
7,vessel_name,str,5,0,EASTERN BRANCH
8,imo,str,1,4,IMO8986949
9,call_sign,str,5,0,WDL7894


### NOAA AIS — Initial Schema Inspection

The January 1, 2026 AIS sample contains **17 columns**. The initial inspection used five records to identify field names, inferred pandas data types, and potential data-quality considerations.

**Source file:** `ais-2026-01-01.csv.zst`  
**Compressed size:** 228.16 MB  
**Format:** Zstandard-compressed CSV  
**Expected grain:** One vessel-position observation at a timestamp

| Column | Observed pandas Type | Description | Initial Findings |
|---|---|---|---|
| `mmsi` | int64 | Maritime Mobile Service Identity; identifies an AIS station | Candidate string identifier; may not uniquely represent a vessel over time |
| `base_date_time` | str | UTC timestamp of AIS observation | Candidate timestamp |
| `longitude` | float64 | Vessel longitude in decimal degrees | Validate range -180 to 180 |
| `latitude` | float64 | Vessel latitude in decimal degrees | Validate range -90 to 90 |
| `sog` | float64 | Speed Over Ground, in knots | May contain null or unusual values |
| `cog` | float64 | Course Over Ground, in degrees | 1 null in sample; distinguish from heading |
| `heading` | float64 | Vessel heading, in degrees | 2 nulls in sample |
| `vessel_name` | str | Reported or enriched vessel name | Potential cross-source matching attribute |
| `imo` | str | International Maritime Organization vessel identifier | Example: `IMO8986949`; 4 nulls |
| `call_sign` | str | Vessel radio call sign | Potential cross-source matching attribute |
| `vessel_type` | int64 | AIS vessel-type classification code | Categorical code, not a measurement |
| `status` | float64 | AIS navigational status code | 1 null; categorical code |
| `length` | int64 | Reported vessel length, generally in meters | Compare with USCG dimensions |
| `width` | int64 | Reported vessel width, generally in meters | Compare with USCG breadth |
| `draft` | float64 | Reported vessel draft, generally in meters | 1 null |
| `cargo` | float64 | AIS cargo/type classification code | 1 null; not cargo weight |
| `transceiver` | str | AIS transceiver class | Example: `A` |

### Preliminary Observations

1. **Identifiers:** `mmsi` was inferred as an integer, but identifiers should generally be stored as strings to preserve formatting and support cross-source matching.

2. **IMO formatting:** NOAA's sample includes the `IMO` prefix. USCG sources may represent IMO identifiers differently, requiring normalization during Silver transformations.

3. **Categorical fields:** `vessel_type`, `status`, and `cargo` contain classification codes rather than numerical measurements.

4. **Missing values:** Several movement and vessel attributes contain nulls. Missing information does not automatically indicate an invalid AIS observation.

5. **Source grain:** Unlike the USCG registry sources, AIS contains repeated vessel observations over time. This must be validated by examining MMSI and timestamp combinations.

### Next Investigation

Profile 10,000 observations to examine:

- Number of unique MMSIs
- Multiple observations per MMSI
- Timestamp uniqueness
- Potential duplicate MMSI/timestamp combinations

**Status:** Step 0B — Source investigation. Data types and canonical mappings remain preliminary.

In [6]:
with open(sample_path, "rb") as f:
    dctx = zstd.ZstdDecompressor()

    with dctx.stream_reader(f) as reader:
        df_profile = pd.read_csv(
            reader,
            nrows=10_000,
            dtype={"mmsi": "string", "imo": "string"}
        )

print("Observations:", len(df_profile))
print("Unique MMSIs:", df_profile["mmsi"].nunique())
print(
    "Unique MMSI/timestamp pairs:",
    df_profile[["mmsi", "base_date_time"]]
    .drop_duplicates()
    .shape[0]
)

display(
    df_profile.groupby("mmsi")
    .agg(
        observations=("base_date_time", "size"),
        unique_timestamps=("base_date_time", "nunique")
    )
    .sort_values("observations", ascending=False)
    .head(10)
)

Observations: 10000
Unique MMSIs: 7503
Unique MMSI/timestamp pairs: 10000


,observations,unique_timestamps
mmsi,,
367533950,3,3
367393650,3,3
367534880,3,3
367660250,3,3
368274610,3,3
368353140,3,3
367374630,3,3
367091160,3,3
367701380,3,3


In [7]:
print("Earliest:", df_profile["base_date_time"].min())
print("Latest:  ", df_profile["base_date_time"].max())

identifier_cols = [
    "mmsi", "imo", "call_sign", "vessel_name"
]

identifier_profile = pd.DataFrame({
    "non_null": df_profile[identifier_cols].notna().sum(),
    "null_count": df_profile[identifier_cols].isna().sum(),
    "null_pct": (
        df_profile[identifier_cols].isna().mean() * 100
    ).round(2),
    "unique_values": df_profile[identifier_cols].nunique()
})

display(identifier_profile)

Earliest: 2026-01-01 00:00:00
Latest:   2026-01-01 00:02:19


,non_null,null_count,null_pct,unique_values
mmsi,10000,0,0.00,7503
imo,3745,6255,62.55,2705
call_sign,9014,986,9.86,6569
vessel_name,9993,7,0.07,7046
